<div align="center">

# Universidad Autónoma de Chihuahua
## Facultad de Ingeniería
### Maestría en Ingeniería en Computación

**Materia:** Machine Learning (MIC1)

**Actividad:** Proyecto Final — Notebook 1: Configuración inicial

**Alumno:** Jonahtan Gabriel González Cardoza
**Matrícula:** 345491

**Catedrática:** Dra. Olanda Prieto Ordaz

**Fecha de elaboración:** 21 de noviembre de 2026

</div>

**Proyecto:** predicción de la situación final de un estudiante (*Dropout*, *Enrolled*, *Graduate*) con el dataset
*Predict Students' Dropout and Academic Success* (UCI 697). Este proyecto reúne, en cinco notebooks reproducibles, el trabajo
de las tres prácticas anteriores (regresión logística del Proyecto F_1, SVM y métodos de ensamble).

**Objetivo de este notebook.** Cargar y limpiar `data.csv`, inspeccionar su estructura y generar **una sola vez** la partición Train/Test (80/20, estratificada, `random_state = 42`). Los notebooks 02–04 leen esa partición desde `outputs/data/`, así que todos usan exactamente los mismos estudiantes.

# II. Introducción

**Dataset.** *Predict Students' Dropout and Academic Success* (Realinho et al., 2022; UCI Machine Learning Repository, id = 697): 4,424 estudiantes de
17 licenciaturas del Instituto Politécnico de Portalegre (Portugal), con 36 características demográficas, socioeconómicas, macroeconómicas y
académicas (inscripción, 1er y 2º semestres).

**Problema.** Clasificación supervisada **multiclase**: predecir la situación del estudiante al final de la duración normal de la carrera.

**Variable objetivo.** `Target`: *Dropout* (0, abandonó), *Enrolled* (1, sigue inscrito sin titularse) o *Graduate* (2, se tituló a tiempo).

**Estructura del proyecto**

| Notebook | Contenido | Produce |
|---|---|---|
| `01_configuracion_inicial` | Carga, inspección y partición Train/Test | `outputs/data/` |
| `02_eda` | EDA (solo Train) y preprocesamiento | `outputs/figs/02_*`, `outputs/tablas/02_*` |
| `03_modelos` | Modelos de referencia, ensambles base y optimización de hiperparámetros (solo Train, CV de 5 folds) | `outputs/modelos/`, `outputs/figs/03_*` |
| `04_resultados` | OOF, CV repetida, **evaluación única en Test**, intervalos de confianza, interpretabilidad y comparación con la literatura | `outputs/figs/04_*`, `outputs/tablas/04_*` |
| `05_ejecucion_completa` | Ejecuta 01 → 04 en orden y verifica que las métricas coincidan con las ya reportadas | `outputs/ejecutados/` |

Las constantes, las métricas y las funciones compartidas están en `config.py`.

# III. Desarrollo

In [ ]:
from config import *
import sys, sklearn, interpret

print("Python:", sys.version.split()[0], "| scikit-learn:", sklearn.__version__, "| pandas:", pd.__version__,
      "| numpy:", np.__version__, "| interpret:", interpret.__version__)
print("RANDOM_STATE =", RANDOM_STATE)

## Fase 1. Obtención e inspección general de los datos

### 1. Obtención de los datos

| Concepto | Valor |
|---|---|
| Nombre del dataset | Predict Students' Dropout and Academic Success |
| Fuente de procedencia | Instituto Politécnico de Portalegre (Portugal); publicado por Realinho et al. (2022) |
| Número de instancias | 4,424 |
| Número de características | 36 (+ `Target`) |
| Variable objetivo | `Target`: *Dropout*, *Enrolled* o *Graduate* al final de la duración normal de la carrera (3 años; 4 en Enfermería) |
| Tipo de problema | Clasificación **multiclase** con 3 clases: Dropout, Enrolled y Graduate |
| Enlace | https://archive.ics.uci.edu/dataset/697/predict+students+dropout+and+academic+success · https://doi.org/10.5281/zenodo.5777339 |

Se carga el `data.csv` del proyecto. El archivo usa `;` como separador, trae un BOM en el primer encabezado (`utf-8-sig`), un tabulador al final del nombre `Daytime/evening attendance` y el typo `Nacionality`; las tres cosas se corrigen al cargar.

In [ ]:
df = cargar_datos()
print("Dimensiones:", df.shape)

### 2. Inspección general antes de la partición
Solo inspección **estructural**: dimensiones, columnas, tipos, primeras filas, faltantes, duplicados, posibles identificadores y formato/distribución general de la variable objetivo. Aquí todavía no se analizan relaciones entre variables.

In [ ]:
print("Dimensiones:", df.shape)
print("\nTipos de datos:\n", df.dtypes.value_counts())
print("\nColumnas:", list(df.columns))
df.head()

In [ ]:
print("Valores faltantes (NaN) en todo el dataset:", df.isna().sum().sum())
print("Registros duplicados:", df.duplicated().sum())

# Posibles identificadores: columnas con un valor distinto en (casi) cada fila
n_unicos = df.nunique().sort_values(ascending=False)
print("\nColumnas con más valores distintos:")
print(n_unicos.head(5))
print("\n¿Alguna columna tiene un valor único por fila (posible ID)?:", bool((n_unicos == len(df)).any()))

In [ ]:
print("Formato de Target:", df["Target"].dtype, "| valores:", sorted(df["Target"].unique()))
print()
print(pd.DataFrame({"n": df["Target"].value_counts(),
                    "proporción": df["Target"].value_counts(normalize=True).round(4)}))
guardar_tabla(pd.DataFrame({"n": df["Target"].value_counts(), "proporción": df["Target"].value_counts(normalize=True).round(4)}), "01_distribucion_target")

**Interpretación.**
- **Dimensiones:** 4,424 filas y 37 columnas (36 características + `Target`). Hay 29 columnas `int64`, 7 `float64` y una de texto (`Target`).
- **Faltantes y duplicados:** no hay `NaN` ni registros duplicados.
- **Identificadores:** ninguna columna tiene un valor distinto por fila (la que más valores distintos tiene es `Curricular units 1st sem (grade)`, con 805 de 4,424), así que no hay un ID que deba excluirse.
- **Tipos aparentes:** todas las características son numéricas, pero varias columnas enteras son códigos de categorías (`Course`, `Application mode`, ocupación de los padres, etc.). Esto se considera en el preprocesamiento (Fase 4).
- **Variable objetivo:** es texto con tres valores: *Graduate* 2,209 (49.93%), *Dropout* 1,421 (32.12%) y *Enrolled* 794 (17.95%). Es un desbalance moderado con una clase claramente minoritaria (*Enrolled*).

En esta fase no se analizaron relaciones entre variables ni se tomaron decisiones basadas en ellas.

## Fase 2. Partición Train/Test

### 3. Definición de X e y
`y` conserva las **tres clases** y se codifica en orden alfabético (el mismo que asigna `LabelEncoder`): **Dropout = 0, Enrolled = 1, Graduate = 2**. `X` son las 36 características restantes.

In [ ]:
X = df.drop(columns="Target")
y = df["Target"].map({clase: i for i, clase in enumerate(CLASES)}).to_numpy()

assert "Target" not in X.columns
assert set(np.unique(y)) == {0, 1, 2}
print("X:", X.shape, " y:", y.shape)
print(pd.crosstab(df["Target"], y, colnames=["y"]))

**Justificación.**
- **No se eliminó ninguna columna:** el dataset no tiene identificadores (Fase 1) y las 36 características son atributos del estudiante o de su contexto.
- **Tres clases en lugar de binarizar:** *Dropout*, *Enrolled* y *Graduate* son desenlaces distintos que piden intervenciones distintas: un estudiante en riesgo de abandonar necesita retención; uno que sigue inscrito al terminar el plazo (rezagado) necesita apoyo para terminar. Es la misma formulación del artículo original del dataset y de los trabajos con los que se compara (Martins et al., 2023; Praptomo et al., 2026). El `crosstab` confirma el mapeo: 1,421 → 0, 794 → 1 y 2,209 → 2.
- **El código numérico no implica orden:** la regresión logística multinomial trata las clases como nominales, así que asignar 0, 1, 2 no le dice al modelo que *Enrolled* esté "entre" *Dropout* y *Graduate*.
- **Limitación:** *Enrolled* es una situación al momento del corte; esos estudiantes todavía podrían titularse o abandonar después.

### 4. Partición Train/Test
División 80% Train / 20% Test. Antes de partir se revisa la distribución de las clases para decidir si se estratifica.

In [ ]:
print("Proporción de cada clase (dataset completo):",
      dict(zip(CLASES, np.bincount(y) / len(y))))

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)
print("Train:", X_train.shape, " Test:", X_test.shape)
print("Estudiantes por clase en Test:", dict(zip(CLASES, np.bincount(y_test))))

**Justificación de la estratificación.** Las clases están desbalanceadas: *Graduate* 49.93%, *Dropout* 32.12% y *Enrolled* 17.95%. Con una partición aleatoria simple, la proporción de la clase minoritaria en Test podría variar por azar y, con solo ≈160 estudiantes *Enrolled* en Test, eso movería bastante las métricas macro. Por eso se usa `stratify=y` con las tres clases. Resultado: 3,539 estudiantes en Train y 885 en Test (284 *Dropout*, 159 *Enrolled* y 442 *Graduate*). Es la misma composición de Test que reporta Praptomo et al. (2026), que también usa una partición 80/20 estratificada con `random_state = 42`.

### 5. Verificación de la partición

In [ ]:
def proporciones(v):
    return pd.Series(np.bincount(v, minlength=3) / len(v), index=CLASES)

verificacion = pd.DataFrame({"Original": proporciones(y), "Train": proporciones(y_train),
                             "Test": proporciones(y_test)})
verificacion.loc["n"] = [len(y), len(y_train), len(y_test)]
verificacion.round(4)
guardar_tabla(verificacion.round(4), "01_verificacion_particion")

**Interpretación.** Las proporciones de las tres clases son prácticamente iguales en el dataset original, en Train y en Test (diferencias ≤ 0.0003): *Dropout* 0.3212 / 0.3213 / 0.3209, *Enrolled* 0.1795 / 0.1794 / 0.1797 y *Graduate* 0.4993 / 0.4993 / 0.4994. Ambos conjuntos son representativos del original. Esta comparación solo verifica la partición; no se usa Test para ninguna decisión.

### 6. Guardado de la partición
La partición se guarda en `outputs/data/train.csv` y `outputs/data/test.csv` (con el índice original de cada estudiante) y la lista de columnas
por tipo en `outputs/data/columnas.json`. **A partir de aquí ningún notebook vuelve a particionar.**

In [ ]:
guardar_particion(X_train, X_test, y_train, y_test)

# Verificación: la partición coincide con la de las prácticas anteriores (mismo estudiante en cada conjunto)
assert X_train.shape == (3539, 36) and X_test.shape == (885, 36), "tamaños distintos a los de las prácticas anteriores"
assert list(np.bincount(y_test)) == [284, 159, 442], "composición de Test distinta a la de las prácticas anteriores"
X_tr2, X_te2, y_tr2, y_te2 = cargar_particion()
assert X_tr2.equals(X_train) and X_te2.equals(X_test) and (y_tr2 == y_train).all() and (y_te2 == y_test).all(), "la partición guardada no coincide"
print("Partición guardada y verificada:", X_train.shape, X_test.shape, "| clases en Test:", dict(zip(CLASES, np.bincount(y_test))))

**Qué produjo este notebook.** `outputs/data/train.csv` (3,539 estudiantes), `outputs/data/test.csv` (885) y `outputs/data/columnas.json`.
Test no se vuelve a abrir hasta la evaluación final del notebook 04.